# Stage 0 - download Zenodo 4266616 and build timelines (CPU runtime is enough)

Run cells top to bottom. If any cell errors, read the full error message; do not edit the loader to be less strict.

In [ ]:
# Kaggle setup. Settings panel: Internet ON for stage 0 only; GPU only for stage 1.
# Add your private dataset "stresstwin-p" via Add Input. The folder that contains pyproject.toml is found automatically.
import os, shutil, subprocess, sys
def find_pkg():
    """Find the package wherever Kaggle mounted the dataset (the mount path differs between Kaggle versions), or unzip it if it was left zipped."""
    import zipfile
    for r, dirs, files in os.walk('/kaggle/input'):
        if 'pyproject.toml' in files and os.path.isdir(os.path.join(r, 'stresstwin_p')):
            return r
    for r, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.zip') and 'stresstwin' in f.lower() and 'reproducibility' not in f.lower():
                out = '/kaggle/working/pkg_unzipped'; zipfile.ZipFile(os.path.join(r, f)).extractall(out)
                for r2, d2, f2 in os.walk(out):
                    if 'pyproject.toml' in f2 and os.path.isdir(os.path.join(r2, 'stresstwin_p')):
                        return r2
    print('Nothing found. What is mounted under /kaggle/input:')
    for r, d, f in os.walk('/kaggle/input'):
        if r.count(os.sep) <= 5: print(' ', r, '| files:', f[:6])
    return None
root = find_pkg()
assert root, 'package not found: in the right panel click Add Input -> Your Datasets -> stresstwin-p (it must be listed under Input), then run again'
if not os.path.exists('/kaggle/working/stresstwin_pkg'): shutil.copytree(root, '/kaggle/working/stresstwin_pkg')
subprocess.run([sys.executable,'-m','pip','-q','install','-e','/kaggle/working/stresstwin_pkg'],check=True)
WORK='/kaggle/working/work'; os.makedirs(WORK, exist_ok=True)
# /kaggle/working is kept only as notebook OUTPUT. After each stage: Save Version, then make the output a PRIVATE dataset.
# Never make a dataset containing texts.csv public.

In [ ]:
# 1) download (CC-BY-4.0). File name, size and md5 are taken from https://zenodo.org/records/4266616
import hashlib, subprocess, os
RAW = f"{WORK}/raw"; os.makedirs(RAW, exist_ok=True)
zp = f"{RAW}/MentalHealthSubreddits2017.zip"
if not os.path.exists(zp):
    subprocess.run(["wget","-q","--show-progress","-O",zp,"https://zenodo.org/records/4266616/files/MentalHealthSubreddits2017.zip?download=1"],check=True)
h = hashlib.md5(open(zp,"rb").read()).hexdigest()
print("md5", h, "OK" if h=="64a61e5883f2dbe80fbfdf173df984f9" else "MISMATCH - do not continue")
assert h=="64a61e5883f2dbe80fbfdf173df984f9"
subprocess.run(["unzip","-q","-o",zp,"-d",RAW],check=True)
for r,_,fs in os.walk(RAW):
    for f in sorted(fs):
        if f.endswith(".pkl"): print(os.path.join(r,f), round(os.path.getsize(os.path.join(r,f))/1e6,1),"MB")

In [ ]:
# 2) secret salt, the SAME in every run. Kaggle: Add-ons -> Secrets -> Add Secret, label STRESSTWIN_SALT, value = a long random phrase (keep a copy offline).
import secrets, pathlib, hashlib
sf = pathlib.Path(WORK)/'salt.bin'
try:
    from kaggle_secrets import UserSecretsClient
    sf.write_bytes(hashlib.sha256(UserSecretsClient().get_secret('STRESSTWIN_SALT').encode()).digest()); print('salt from the Kaggle secret (reproducible)')
except Exception as e:
    if not sf.exists(): sf.write_bytes(secrets.token_bytes(32))
    print('WARNING: no STRESSTWIN_SALT secret; random salt, not reproducible', repr(e)[:80])
print('salt ok', sf.stat().st_size, 'bytes')

In [ ]:
# 3) build tables. A SchemaError prints the keys actually found; compare them with the expected schema.
!python -m stresstwin_p.timeline_builder --folder {WORK}/raw --salt-file {WORK}/salt.bin --out {WORK}/tables

In [ ]:
import pandas as pd
o = pd.read_csv(f'{WORK}/tables/observations.csv'); print(o.groupby('community').author.nunique())
g = o.groupby('author').agg(n=('t_days','size'), span=('t_days', lambda s: s.max()-s.min()))
print((g.n>=5).sum(), 'authors with >=5 obs;', ((g.n>=5)&(g.span>=14)).sum(), 'also spanning >=14 days')
# DECISION RULE: fewer than ~40 eligible authors in a community => it cannot be a held-out community. Report it, do not hide it.